# 🚢 Task 1 – Data Cleaning & Preprocessing (Titanic Dataset)

**Author:** Your Name  
**Dataset:** Titanic Passenger Data  
**Goal:** Take raw, messy data → clean, structured, ML-ready data

---

## Problem Statement
Real-world datasets are always messy — missing values, wrong data types, duplicate rows, and inconsistent column names. Before any analysis or machine learning, we must clean the data. In this project, we clean the Titanic dataset, which contains information about 891 passengers and whether they survived.

---

## ✅ Step 0: Import Libraries

In [ ]:
# pandas  → data manipulation and cleaning
# numpy   → numerical operations (used for NaN handling)
import pandas as pd
import numpy as np

print("✅ Libraries imported successfully")
print(f"   pandas version : {pd.__version__}")
print(f"   numpy  version : {np.__version__}")

---
## ✅ Step 1: Load the Dataset

In [ ]:
# Load the raw Titanic CSV file
# Source: https://github.com/datasciencedojo/datasets/blob/master/titanic.csv
df = pd.read_csv("titanic.csv")

print(f"📦 Dataset loaded successfully!")
print(f"   Rows    : {df.shape[0]}")
print(f"   Columns : {df.shape[1]}")
print()

# Preview the first 5 rows
df.head()

---
## ✅ Step 2: Explore the Dataset (Before Cleaning)

Before cleaning anything, we must **understand** what we're working with. Two key tools:
- `df.info()` → column names, non-null counts, data types
- `df.describe()` → statistical summary (mean, std, min, max)

In [ ]:
# df.info() tells us:
#  - How many non-null values each column has (missing = total - non-null)
#  - The data type of each column
#  - Memory usage

print("=" * 50)
print("📋 Dataset Info (BEFORE Cleaning)")
print("=" * 50)
df.info()

In [ ]:
# df.describe() gives us statistical summary for all numeric columns
# Key observations:
#  - Age: count=714 → 177 missing values
#  - Survived: mean=0.38 → only 38% survived
#  - Fare: ranges from 0 to 512 → large spread

print("=" * 50)
print("📊 Statistical Summary (BEFORE Cleaning)")
print("=" * 50)
df.describe()

---
## ✅ Step 3: Check Missing Values

Identify which columns have missing data and how much.

In [ ]:
# Count missing values in each column
missing_count = df.isnull().sum()

# Calculate percentage of missing values
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)

# Combine into a clear summary table
missing_df = pd.DataFrame({
    'Missing Count': missing_count,
    'Missing %': missing_pct
})

# Show only columns that have missing values
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing %', ascending=False)

print("🔍 Columns with Missing Values:")
print()
print(missing_df)
print()
print("📌 Observation:")
print("   • Cabin    → 77.1% missing → should DROP this column")
print("   • Age      → 19.9% missing → fill with median")
print("   • Embarked →  0.2% missing → fill with mode (most common value)")

---
## ✅ Step 4: Handle Missing Values

**Strategy:**
| Column | Missing % | Action | Reason |
|--------|-----------|--------|---------|
| `Cabin` | 77.1% | **Drop** | Too many missing; not reliable |
| `Age` | 19.9% | **Fill with median** | Median is robust to outliers |
| `Embarked` | 0.2% | **Fill with mode** | Only 2 rows missing; use most frequent |


In [ ]:
# --- Handle Age ---
# We use MEDIAN instead of mean because Fare and Age have outliers
# Median is not skewed by extreme values
age_median = df['Age'].median()
print(f"Age median value: {age_median}")
df['Age'] = df['Age'].fillna(age_median)
print(f"✅ Age: filled {df['Age'].isnull().sum()} remaining missing values (was 177)")

# --- Drop Cabin ---
# 77% missing is too high to fill meaningfully
df.drop(columns=['Cabin'], inplace=True)
print("✅ Cabin: column dropped")

# --- Handle Embarked ---
# Mode = most frequent value ('S' = Southampton)
embarked_mode = df['Embarked'].mode()[0]
print(f"Embarked mode value: '{embarked_mode}'")
df['Embarked'] = df['Embarked'].fillna(embarked_mode)
print(f"✅ Embarked: filled missing values")

print()
print("🔍 Missing values AFTER handling:")
print(df.isnull().sum())

---
## ✅ Step 5: Remove Duplicate Rows

In [ ]:
# Check for duplicate rows
dupes_before = df.duplicated().sum()
print(f"🔍 Duplicate rows found: {dupes_before}")

# Remove duplicates (keeps first occurrence)
df.drop_duplicates(inplace=True)

print(f"✅ Duplicates removed. Rows after: {len(df)}")

---
## ✅ Step 6: Convert Data Types

Machine learning models work with **numbers only**. We need to convert:
- `Sex`: `male` → `0`, `female` → `1`
- `Embarked`: `S` → `0`, `C` → `1`, `Q` → `2`

In [ ]:
# --- Convert Sex column ---
# male = 0, female = 1
print("Sex column before:")
print(df['Sex'].value_counts())

df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})

print("\nSex column after (0=male, 1=female):")
print(df['Sex'].value_counts())

print()

# --- Convert Embarked column ---
# S=Southampton=0, C=Cherbourg=1, Q=Queenstown=2
print("Embarked column before:")
print(df['Embarked'].value_counts())

df['Embarked'] = df['Embarked'].map({'S': 0, 'C': 1, 'Q': 2})

print("\nEmbarked column after (0=S, 1=C, 2=Q):")
print(df['Embarked'].value_counts())

print()

# --- Ensure correct numeric types ---
df['Age']      = df['Age'].astype(float)
df['Fare']     = df['Fare'].astype(float)
df['Survived'] = df['Survived'].astype(int)

print("✅ Data types after conversion:")
print(df.dtypes)

---
## ✅ Step 7: Rename Columns

Clean, consistent column names (snake_case) are easier to work with in code.

In [ ]:
# Rename columns to snake_case and descriptive names
print("Columns BEFORE renaming:")
print(list(df.columns))

df.rename(columns={
    'PassengerId' : 'passenger_id',
    'Survived'    : 'survived',
    'Pclass'      : 'passenger_class',
    'Name'        : 'name',
    'Sex'         : 'sex',
    'Age'         : 'age',
    'SibSp'       : 'siblings_spouses',
    'Parch'       : 'parents_children',
    'Ticket'      : 'ticket',
    'Fare'        : 'fare',
    'Embarked'    : 'embarked'
}, inplace=True)

print("\nColumns AFTER renaming:")
print(list(df.columns))
print("\n✅ Columns renamed successfully")

---
## ✅ Step 8: Before vs After Summary

In [ ]:
print("="*55)
print("         BEFORE vs AFTER CLEANING SUMMARY")
print("="*55)

summary = pd.DataFrame({
    'Metric': [
        'Total Rows',
        'Total Columns',
        'Missing Values (Age)',
        'Missing Values (Cabin)',
        'Missing Values (Embarked)',
        'Duplicate Rows',
        'Sex dtype',
        'Embarked dtype',
        'Column names style'
    ],
    'Before': [
        891, 12, 177, 687, 2, 0,
        'object (text)',
        'object (text)',
        'PascalCase / abbreviations'
    ],
    'After': [
        len(df), len(df.columns), 0, 'DROPPED', 0, 0,
        'int64 (0/1)',
        'int64 (0/1/2)',
        'snake_case / descriptive'
    ]
})

print(summary.to_string(index=False))

print()
print("✅ Final missing values check:")
print(df.isnull().sum())

In [ ]:
# Final preview of the clean dataset
print("\n🎯 Clean dataset preview:")
df.head(10)

In [ ]:
# Final statistical summary after cleaning
print("📊 Statistical Summary (AFTER Cleaning)")
df.describe()

---
## ✅ Step 9: Save the Cleaned Dataset

In [ ]:
# Save cleaned dataset as a new CSV file
# index=False → don't write the row numbers as a column
df.to_csv("cleaned_titanic.csv", index=False)

print("🎉 Cleaned dataset saved as 'cleaned_titanic.csv'")
print(f"   Final shape: {df.shape[0]} rows × {df.shape[1]} columns")
print("   Zero missing values ✅")
print("   All numeric columns ✅")
print("   Clean column names ✅")
print("\n🚀 Dataset is ready for analysis or machine learning!")

---
## 📝 Summary of Steps Performed

| Step | Action | Detail |
|------|--------|--------|
| 1 | **Load Data** | Loaded raw Titanic CSV with 891 rows and 12 columns |
| 2 | **Explore** | Used `df.info()` and `df.describe()` to understand the data |
| 3 | **Check Missing** | Found Age (177), Cabin (687), Embarked (2) missing values |
| 4 | **Handle Missing** | Age → median fill; Cabin → dropped; Embarked → mode fill |
| 5 | **Remove Duplicates** | Checked and removed duplicate rows |
| 6 | **Convert Types** | Sex → 0/1 numeric; Embarked → 0/1/2 numeric |
| 7 | **Rename Columns** | Applied snake_case descriptive column names |
| 8 | **Verify** | Confirmed 0 missing values, correct types |
| 9 | **Save** | Exported `cleaned_titanic.csv` |

---
**Tools Used:** Python, pandas, numpy, Jupyter Notebook  
**Dataset Source:** https://github.com/datasciencedojo/datasets/blob/master/titanic.csv